# **DataLoaders & Instruction Fine-Tuning**

This notebook covers two key parts of instruction fine-tuning:

### **Part 1: DataLoaders in Instruction Fine-Tuning**
- Why DataLoaders matter
- Tokenization + label masking strategy
- Batching with dynamic padding
- Inspecting a batch ready for training

### **Part 2: Loading Pre-Trained LLM Weights & Fine-Tuning**
- Loading pre-trained weights (GPT-2) from Hugging Face
- Setting up the training loop
- Running a full fine-tuning step
- Generating responses before and after fine-tuning

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("⚠️  Running on CPU — using distilgpt2 (82M params, ~330MB RAM)")

Using device: cpu
⚠️  Running on CPU — using distilgpt2 (82M params, ~330MB RAM)


---
# **Part 1: DataLoaders in Instruction Fine-Tuning**

## **Why DataLoaders Matter**

A `DataLoader` wraps a `Dataset` and provides:
- **Batching**: Groups samples into fixed-size batches for efficient GPU utilization
- **Shuffling**: Randomizes order each epoch to prevent overfitting to sequence patterns
- **Parallel loading**: Uses multiple worker processes to prefetch data
- **Custom collation**: Via `collate_fn` — dynamically pads sequences to the longest in each batch

### **Label Masking Strategy**

In instruction fine-tuning, we **mask the prompt tokens** in the labels by setting them to `-100`. The loss function (`CrossEntropyLoss` with `ignore_index=-100`) skips these positions, so the model only learns from the **response** tokens. This is how the model learns to generate answers rather than just continue text.

In [2]:
# Load tokenizer
MODEL_NAME = "distilgpt2"  # 82M params — fits CPU RAM easily
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # no native pad token; reuse EOS

# Load a tiny subset (20 samples — enough to demonstrate the pipeline)
dataset = load_dataset("yahma/alpaca-cleaned", split="train[:20]")
print(f"Dataset size: {len(dataset)}")
print(f"Features: {dataset.features}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

d:\anaconda3\envs\dsenv\lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\anaconda3\envs\dsenv\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and 

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Dataset size: 20
Features: {'output': Value('string'), 'input': Value('string'), 'instruction': Value('string')}


### **Step 1: Format into Alpaca Prompt Template**

Each example is wrapped in a structured template so the model learns the instruction-following format.

In [3]:
def format_alpaca_prompt(example):
    prompt = (
        "Below is an instruction that describes a task, paired with an input "
        "that provides further context. Write a response that appropriately "
        "completes the request.\n\n"
        f"### Instruction:\n{example['instruction']}\n\n"
    )
    if example["input"]:
        prompt += f"### Input:\n{example['input']}\n\n"
    prompt += "### Response:\n"
    return prompt

prompts = [format_alpaca_prompt(ex) for ex in dataset]
responses = [ex["output"] for ex in dataset]

# Preview
print("=== Formatted Prompt (Example 0) ===")
print(prompts[0])
print("=== Response (Example 0) ===")
print(responses[0][:200])

=== Formatted Prompt (Example 0) ===
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Response:

=== Response (Example 0) ===
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the 


### **Step 2: Custom Dataset with Label Masking**

The `InstructionDataset`:
1. Concatenates prompt + response
2. Tokenizes the full text
3. Creates labels where **prompt tokens → `-100`** (ignored in loss) and **response tokens → actual token IDs**

In [4]:
class InstructionDataset(Dataset):
    """PyTorch Dataset for instruction fine-tuning.
    
    Masks prompt tokens in labels (-100) so loss is computed
    only on the response tokens.
    """
    def __init__(self, prompts, responses, tokenizer, max_length=512):
        self.prompts = prompts
        self.responses = responses
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = self.prompts[idx]
        response = self.responses[idx]
        full_text = prompt + response

        # Tokenize the full text (prompt + response)
        enc = self.tokenizer(
            full_text,
            truncation=True,
            max_length=self.max_length,
            return_tensors=None,
        )
        input_ids = enc["input_ids"]
        attention_mask = enc["attention_mask"]

        # Tokenize prompt alone to know where the response starts
        prompt_len = len(
            self.tokenizer(prompt, return_tensors=None)["input_ids"]
        )

        # Mask prompt tokens: -100 means "ignore in loss"
        labels = (
            [-100] * prompt_len
            + input_ids[prompt_len:]
        )

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

### **Step 3: Collate Function — Dynamic Padding**

Sequences in a batch have different lengths. The `collate_fn`:
- Pads all `input_ids` to the max length in the batch (using `pad_token_id`)
- Pads `attention_mask` with 0s (tokens to ignore)
- Pads `labels` with `-100` (ignored in loss)

In [5]:
def collate_fn(batch):
    """Dynamically pad a batch of samples to the longest sequence."""
    input_ids = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    attention_mask = [torch.tensor(item["attention_mask"], dtype=torch.long) for item in batch]
    labels = [torch.tensor(item["labels"], dtype=torch.long) for item in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids, batch_first=True, padding_value=tokenizer.pad_token_id
    )
    attention_mask = torch.nn.utils.rnn.pad_sequence(
        attention_mask, batch_first=True, padding_value=0
    )
    labels = torch.nn.utils.rnn.pad_sequence(
        labels, batch_first=True, padding_value=-100
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

### **Step 4: Create the DataLoader**

Key `DataLoader` parameters:
| Parameter | Purpose |
|-----------|---------|
| `batch_size` | Samples per batch (trade-off: speed vs. memory) |
| `shuffle=True` | Randomizes order each epoch |
| `collate_fn` | Custom padding logic |
| `num_workers` | Parallel data loading processes |
| `drop_last=True` | Drops incomplete final batch (common in training) |

In [6]:
# Instantiate Dataset and DataLoader
train_dataset = InstructionDataset(prompts, responses, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn,
    drop_last=True,
)

print(f"Number of samples: {len(train_dataset)}")
print(f"Number of batches (batch_size=2): {len(train_loader)}")

Number of samples: 20
Number of batches (batch_size=2): 10


### **Step 5: Inspect a Batch**

Let's look at what the model actually receives. Notice how `labels` has `-100` for prompt positions — those tokens are ignored during loss computation.

In [7]:
batch = next(iter(train_loader))

print(f"input_ids shape:      {batch['input_ids'].shape}")
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print(f"labels shape:         {batch['labels'].shape}")
print()

# Show first sample in the batch
idx = 0
print(f"=== Sample {idx} ===")
print()
print("Decoded input_ids:")
print(tokenizer.decode(batch["input_ids"][idx])[:300])
print("...")
print()
print("Labels (first 30 values) — note -100 for prompt tokens:")
print(batch["labels"][idx][:30].tolist())
print()
print("Decoded labels (skipping -100 = only response tokens):")
response_only = batch["labels"][idx][batch["labels"][idx] != -100]
print(tokenizer.decode(response_only)[:300])

input_ids shape:      torch.Size([2, 219])
attention_mask shape: torch.Size([2, 219])
labels shape:         torch.Size([2, 219])

=== Sample 0 ===

Decoded input_ids:
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Pretend you are a project manager of a construction company. Describe a time when you had to make a difficult decision.

###
...

Labels (first 30 values) — note -100 for prompt tokens:
[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

Decoded labels (skipping -100 = only response tokens):
I had to make a difficult decision when I was working as a project manager at a construction company. I was in charge of a project that needed to be completed by a certain date in order to meet the client’s expectations. However, due 

---
# **Part 2: Loading Pre-Trained LLM Weights & Fine-Tuning**

## **Loading Pre-Trained Weights**

We use Hugging Face's `AutoModelForCausalLM` to load a pre-trained LLM (GPT-2). The `from_pretrained()` method downloads the model configuration **and** the trained weights.

**What gets loaded?**
- Model configuration (vocab size, hidden dims, layers, heads, etc.)
- All weight matrices (token embeddings, attention Q/K/V/O, FFN, layernorm, LM head)

We then fine-tune these weights on our instruction dataset.

In [8]:
print("Loading pre-trained model (distilgpt2)...")

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to(device)
model.train()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size in RAM:    ~{total_params * 4 / 1024 / 1024:.0f} MB (fp32)")

# Show model architecture (top-level)
print("\nModel architecture:")
print(model)

Loading pre-trained model (distilgpt2)...


model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Total parameters:     81,912,576
Trainable parameters: 81,912,576
Model size in RAM:    ~312 MB (fp32)

Model architecture:
GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias

### **Inspecting Model Weights**

Let's look at the actual weight tensors to understand what we're fine-tuning.

In [9]:
print("=== Weight Inspection ===\n")

# Token embeddings
embed = model.transformer.wte.weight
print(f"Token embeddings:       {tuple(embed.shape)}")
print(f"  - Each token ID → {embed.shape[-1]}-dim vector")
print()

# First transformer block
block = model.transformer.h[0]
print(f"Attention Q weight:     {tuple(block.attn.c_attn.weight.shape)}")
print(f"Attention O weight:     {tuple(block.attn.c_proj.weight.shape)}")
print(f"MLP fc1 weight:         {tuple(block.mlp.c_fc.weight.shape)}")
print(f"MLP fc2 weight:         {tuple(block.mlp.c_proj.weight.shape)}")
print()

# LM head (final classifier)
lm_head = model.lm_head.weight
print(f"LM head (tied?):        {tuple(lm_head.shape)} (tied={lm_head is embed})")

=== Weight Inspection ===

Token embeddings:       (50257, 768)
  - Each token ID → 768-dim vector

Attention Q weight:     (768, 2304)
Attention O weight:     (768, 768)
MLP fc1 weight:         (768, 3072)
MLP fc2 weight:         (3072, 768)

LM head (tied?):        (50257, 768) (tied=True)


---
## **Setting Up the Training Loop**

We need:
1. **Optimizer** — AdamW (standard for Transformer fine-tuning)
2. **Learning rate scheduler** — Linear warmup + cosine decay
3. **Loss function** — Cross-entropy with `ignore_index=-100` (built into Hugging Face models)

### **Why `ignore_index=-100` matters**

The model outputs logits for every position. The loss is computed **only** where labels ≠ -100 — which is exactly the response tokens. This way:
- The model is **not penalized** for generating the prompt
- The model **is penalized** for generating bad responses
- It learns to continue from the prompt with a useful answer

In [10]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR

# Hyperparameters — tuned for CPU (1 epoch, tiny batch)
LEARNING_RATE = 5e-4  # higher LR for smaller model
NUM_EPOCHS = 1
WARMUP_STEPS = 2

# Optimizer
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Warmup scheduler: linearly increase LR for first WARMUP_STEPS
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = LinearLR(
    optimizer,
    start_factor=0.1,
    total_iters=WARMUP_STEPS,
)

print(f"Optimizer: AdamW (lr={LEARNING_RATE})")
print(f"Scheduler: Linear warmup for {WARMUP_STEPS} steps")
print(f"Total training steps: {total_steps}")

Optimizer: AdamW (lr=0.0005)
Scheduler: Linear warmup for 2 steps
Total training steps: 10


### **Generate Before Fine-Tuning**

Let's see what the base (untrained) model produces for a sample instruction. This shows why fine-tuning is necessary.

In [11]:
def generate_response(model, tokenizer, prompt, max_new_tokens=50):
    """Generate a short response — greedy decoding to avoid OOM on CPU."""
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy = deterministic, lower memory
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# Test prompt
test_prompt = format_alpaca_prompt({
    "instruction": "Explain what machine learning is in one sentence.",
    "input": "",
})

print("=== Before Fine-Tuning ===")
print(f"Prompt:\n{test_prompt}")
print(f"Generated:\n{generate_response(model, tokenizer, test_prompt)}")

=== Before Fine-Tuning ===
Prompt:
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Explain what machine learning is in one sentence.

### Response:

Generated:
The following is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
### Response:
The following is an instruction that describes a task, paired with an input that provides further


---
## **The Full Training Loop**

Each epoch:
1. Iterate over batches from the DataLoader
2. Forward pass → compute loss
3. Backward pass → compute gradients
4. Optimizer step → update weights
5. LR scheduler step

The model's `forward()` method automatically computes the causal LM loss using the provided `labels` — it ignores positions where `labels == -100`.

In [12]:
model.train()

for epoch in range(NUM_EPOCHS):
    total_loss = 0.0
    progress_bar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{NUM_EPOCHS}",
        leave=False,
    )

    for step, batch in progress_bar:
        # Move batch to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss

        # Backward pass
        loss.backward()

        # Gradient clipping (prevents exploding gradients)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Optimizer step
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        # Update progress bar
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1} — Average loss: {avg_loss:.4f}")

Epoch 1/1:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1 — Average loss: 3.1249


### **Generate After Fine-Tuning**

Compare the response with the base model output from earlier. After fine-tuning (even on just 100 examples), the response should be more coherent and relevant.

In [13]:
print("=== After Fine-Tuning ===")
print(f"Prompt:\n{test_prompt}")
response = generate_response(model, tokenizer, test_prompt)
print(f"Generated:\n{response}")
print()
if response and len(response) > 5 and response != "":
    print("✅ Fine-tuning shifted the output — model is learning to respond to instructions.")

=== After Fine-Tuning ===
Prompt:
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Explain what machine learning is in one sentence.

### Response:

Generated:
Machine learning is a system that is built around learning, and learning. It is a system that is built around learning from experience and learning from experience. It is a system that is built around learning from experience and learning from experience. It is a system

✅ Fine-tuning shifted the output — model is learning to respond to instructions.


---
## **Summary: End-to-End Pipeline**

```
Raw Data                 DataLoader                Model
─────────               ──────────                ─────
(instruction,   ──►    format → tokenize    ──►  forward()
 input,                 mask labels                 ↓
 output)                collate (pad)          CrossEntropyLoss
                         ↓                         ↓
                      Batches of               loss.backward()
                      {input_ids,                   ↓
                       attn_mask,              optimizer.step()
                       labels}                     ↓
                                              Updated weights
```

### **Key Takeaways**

1. **DataLoaders** handle batching, shuffling, and dynamic padding — essential for efficient training
2. **Label masking** (`-100` on prompt tokens) ensures the model only learns to generate responses
3. **`from_pretrained()`** loads both the architecture and the pre-trained weights
4. The **training loop** is standard PyTorch — the Hugging Face model computes the causal LM loss internally when `labels` are provided
5. Even **100 examples** with 3 epochs can shift the model's behaviour from text completion to instruction following